In [ ]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# %matplotlib qt
%matplotlib widget

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from CustomLogger import CustomLogger as Logger

from dashsrc.plot_components.plot_wrappers.data_selection import group_filter_data

from analytics_processing.modality_loading import session_modality_from_nas
from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames

# import GridSpecFromSubplotSpec
from matplotlib.gridspec import GridSpecFromSubplotSpec
import scipy.stats as stats
import statsmodels.api as sm
from matplotlib.colors import LinearSegmentedColormap, Normalize



In [ ]:
output_dir = "./outputs/experimental/"
output_dir = "/mnt/SpatialSequenceLearning/Simon/amit_analysis"
data = {}
nas_dir = C.device_paths()[0]
Logger().init_logger(None, None, logging_level="DEBUG")

In [ ]:
animal_ids = [6]
paradigm_ids = [1100]
session_ids = None

In [ ]:
session_dirs, _ = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)
session_names = fullfnames2snames(session_dirs)

In [ ]:


fr_raw = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
fr = fr_raw.drop("from_ephys_timestamp", axis=1)
fr.index = fr.index.droplevel((0,1,3, ))
fr.set_index('to_ephys_timestamp', append=True, inplace=True)
fr['global_t'] = np.arange(40_000, 40_000*len(fr)+1, 40_000)
fr.set_index('global_t', append=True, inplace=True)
fr.index = fr.index.rename(['session_id', 'session_t', 'global_t'])
fr

In [ ]:
behavior_glm_input = analytics.get_analytics('Behavior40msAligned', session_names=session_names)
# drop index level
behavior_glm_input.index = behavior_glm_input.index.droplevel(['entry_id', 'paradigm_id', 'animal_id'])
drop_cols = ['trial_id',
            #  'cue',
            #  'trial_outcome',
            #  'choice_R1',
            #  'choice_R2',
             
             'frame_pc_timestamp',
             'from_ephys_timestamp',
             'to_ephys_timestamp',
             ]
behavior_glm_input.drop(drop_cols, axis=1, inplace=True)
# turn categorical column into int
behavior_glm_input['track_zone'] = behavior_glm_input['track_zone'].cat.codes

# set low liklihood head angle detection
mask = behavior_glm_input['facecam_pose_nose_neck_body1_angle_likelihood'] > 0
behavior_glm_input.loc[~mask, 'facecam_pose_nose_neck_body1_angle'] = np.nan
behavior_glm_input.loc[~mask, 'facecam_pose_nose_neck_body1_angle_velocity'] = np.nan
behavior_glm_input = behavior_glm_input.rename(columns={
    'facecam_pose_nose_neck_body1_angle': 'head_angle', 
    'facecam_pose_nose_neck_body1_angle_velocity': 'head_angle_velocity'
})
behavior_glm_input = behavior_glm_input.drop('facecam_pose_nose_neck_body1_angle_likelihood', axis=1)


# choice columns should be boolean
behavior_glm_input = behavior_glm_input.astype({'choice_R1': 'float', 'choice_R2': 'float'})
behavior_glm_input['trial_outcome'] = behavior_glm_input['trial_outcome'].clip(0, 1)
behavior_glm_input['global_t'] = fr.index.get_level_values('global_t')
behavior_glm_input

In [ ]:
plt.close("all")
n_features = len(behavior_glm_input.columns)
n_cols = 5
n_rows = int(np.ceil(n_features / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3*n_rows))
axes = axes.flatten()  # Flatten to easily iterate

for i, col in enumerate(behavior_glm_input.columns):
    print(col)
    axes[i].hist(behavior_glm_input[col].dropna(), bins=100)
    axes[i].set_title(col, fontsize=10)
    axes[i].tick_params(labelsize=8)

# Hide extra subplots if any
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
session_ids = behavior_glm_input.index.unique(0)[:8]
session_ids

In [ ]:
# more preprocessing
# Filter out rows with NaN values
print(behavior_glm_input.notna().all(axis=1))
nonan_mask = behavior_glm_input.notna().all(axis=1)
behavior_glm_input = behavior_glm_input[nonan_mask]

# first sessions (double rewards)
behavior_glm_input = behavior_glm_input.loc[session_ids]

# Drop highly correlated features (need to reassign or use inplace=True)
# correlated_features = ['frame_raw', 
#                        'frame_raw_quantile',
#                        'post_lick',
#                        'frame_abs_acceleration', 
#                        ]


# Only drop columns that actually exist in the dataframe
# existing_cols_to_drop = [col for col in correlated_features if col in behavior_glm_input.columns]
# behavior_glm_input = behavior_glm_input.drop(existing_cols_to_drop, axis=1)

# drop 'zone' columns
zone_columns = [col for col in behavior_glm_input.columns if 'zone_' in col]
behavior_glm_input = behavior_glm_input.drop(zone_columns, axis=1)

print(f"Final shape: {behavior_glm_input.shape}")
behavior_glm_input

In [ ]:
# Compute Pearson correlation matrix between all features
correlation_matrix = behavior_glm_input.corr(method='pearson')

# Create heatmap
plt.figure(figsize=(14, 12))
im = plt.imshow(correlation_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)

# Add colorbar
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label('Pearson Correlation', fontsize=12)

# Set ticks and labels
feature_names = correlation_matrix.columns
plt.xticks(range(len(feature_names)), feature_names, rotation=90, fontsize=9)
plt.yticks(range(len(feature_names)), feature_names, fontsize=9)

# Add grid for better readability
plt.grid(False)

# Add title
plt.title('Pearson Correlation Matrix of Behavioral Features', fontsize=14, pad=20)

plt.tight_layout()
plt.show()

# Optionally, print the correlation matrix
print("\nCorrelation Matrix:")
print(correlation_matrix.round(3))

In [ ]:
def add_joint_plot(fig, parent_spec, x, y, color='gray', alpha=0.01, share_x_with=None, share_y_with=None):
    """Creates a joint scatter plot (no marginals) inside a subplot grid cell."""
    # Convert to numpy arrays
    x_np = np.array(x, dtype=np.float64) # behavior variable, regresser
    y_np = np.array(y, dtype=np.float64) # firing rate
    
    # Create mask for valid values
    valid_mask = ~(np.isnan(x_np) | np.isnan(y_np))
    x_clean = x_np[valid_mask]
    y_clean = y_np[valid_mask]

    # Create subplot and share axes if requested
    gs = GridSpecFromSubplotSpec(1, 1, subplot_spec=parent_spec)
    ax_joint = fig.add_subplot(gs[0, 0], sharex=share_x_with, sharey=share_y_with)

    # Plot data, y the firing rate is on the x axis, x the behavior regresser is on the y axis
    # add jitter to x for better visualization
    # jitter_strength = 0.1 * (np.max(y_clean) - np.min(y_clean))
    # y_jittered = y_clean + np.random.uniform(-jitter_strength, jitter_strength, size=y_clean.shape)
    y_jittered = y_clean
    ax_joint.scatter(y_jittered, x_clean, s=6, color=color, alpha=alpha)
    
    # jitter the other
    # jitter_strength = 0.05 * (np.max(x_clean) - np.min(x_clean))
    # x_jittered = x_clean + np.random.uniform(-jitter_strength, jitter_strength, size=x_clean.shape)
    x_jittered = x_clean
    ax_joint.scatter(y_jittered, x_jittered, s=6, color=color, alpha=alpha)

    # Turn off ticks
    ax_joint.tick_params(labelleft=False, labelbottom=False)
    [ax_joint.spines[side].set_visible(False) for side in ['top', 'right', 'left', 'bottom']]
    
    return ax_joint

# ==============================================================
# === GLM FITTING AND VISUALIZATION ============================
# ==============================================================

def plot_neuron_session_histograms(behavior_aligned, ):
    session_names = [f"S{s_id:02}" for s_id in behavior_aligned.index.unique('session_id')]
    n_sessions = len(session_names)
    n_regressers = behavior_aligned.shape[1]

    # Create figure with specific layout
    fig = plt.figure(figsize=(1.5 * n_regressers, 1.5 * n_regressers))
    
    # Create GridSpec with extra row/column for histograms
    outer_gs = plt.GridSpec(
        nrows=n_regressers + 1, 
        ncols=n_regressers + 1, 
        figure=fig,
        wspace=0.2, hspace=0.2,
        width_ratios=[0.5] + [1]*n_regressers,  # Make first column narrower
        height_ratios=[0.5] + [1]*n_regressers  # Make first row shorter
    )

    # === LEFT COLUMN: behavior histograms ===
    left_hist_axes = []
    for row, regr_name in enumerate(behavior_aligned.columns):
        print(regr_name)
        ax = fig.add_subplot(outer_gs[row + 1, 0])
        left_hist_axes.append(ax)
        regr_data = behavior_aligned[regr_name]
        
        # Calculate density for consistent y-limits across row
        counts, bins, _ = ax.hist(regr_data, bins=20, density=True, 
                                orientation="horizontal", color='gray', alpha=0.6)
        
        ax.set_ylabel(f"{regr_name}", fontsize=13)
        ax.set_xscale('log')
        ax.grid(axis='x', linestyle='--', alpha=0.4)
        [ax.spines[side].set_visible(False) for side in ['top', 'bottom', 'right']]
        
        # Store density limits for sharing
        ax.density_limits = (min(counts[counts > 0]), max(counts))
        
        if row == n_regressers - 1:
            ax.tick_params(labelsize=7, labelbottom=True)
            ax.set_ylabel("Density", fontsize=6)
        else:
            plt.setp(ax.get_xticklabels(), visible=False)
            
        # Reverse x-axis for better visualization
        xlim = ax.get_xlim()
        ax.set_xlim(xlim[1], xlim[0])

    # === TOP ROW: behavior again ===
    # === TOP ROW: behavior histograms ===
    top_row_axes = []
    shared_ylim = None
    
    for col, regr_name in enumerate(behavior_aligned.columns):
        ax = fig.add_subplot(outer_gs[0, col + 1])
        top_row_axes.append(ax)
        regr_data = behavior_aligned[regr_name]
        
        # Calculate density with same bins as left column
        counts, bins, _ = ax.hist(regr_data, bins=20, density=True, 
                                orientation="vertical", color='gray', alpha=0.6)
        
        ax.set_xlabel(f"{regr_name}", fontsize=13)
        ax.set_yscale('log')
        ax.grid(axis='y', linestyle='--', alpha=0.4)
        [ax.spines[side].set_visible(False) for side in ['top', 'right', 'left']]
        
        # Share y limits across all top histograms
        if shared_ylim is None:
            shared_ylim = (min(counts[counts > 0]), max(counts))
        ax.set_ylim(shared_ylim)
        ax.set_title(f"{regr_name}", fontsize=10)
        
        # Only show x ticks for rightmost histogram
        if col == n_regressers - 1:
            ax.tick_params(labelsize=7, labelleft=True)
            ax.set_ylabel("Density", fontsize=6)
        else:
            plt.setp(ax.get_yticklabels(), visible=False)
    
    # === MAIN GRID: joint plots ===
    # get min and max of behavior_aligned for normalization
    
    for row, regr_name in enumerate(behavior_aligned.columns):
        print("Drawing regresser:", regr_name, '... ')
        regr_data = behavior_aligned[regr_name].values
        if len(regr_data) > 1_000:
                regr_data = np.random.choice(regr_data, size=1_000, replace=False)
        for col, regr_name_2 in enumerate(behavior_aligned.columns):
            regr_data_2 = behavior_aligned[regr_name_2].values
            if len(regr_data_2) > 1_000:
                regr_data_2 = np.random.choice(regr_data_2, size=1_000, replace=False)
            
            _ = add_joint_plot(fig, outer_gs[row + 1, col + 1], 
                               regr_data, regr_data_2,
                               share_x_with=top_row_axes[col],
                               share_y_with=left_hist_axes[row])
            # break
        # break
    fig.suptitle(f"Behavior regressers", fontsize=16)
        
    # set left and right margin to 0.05
    plt.subplots_adjust(left=0.05, right=0.95)
    # plt.savefig(f"../outputs/glm_results/neuron_{neuron_i}_GLM.png", dpi=300)
    plt.show()
    # plt.close()

plt.close("all")
plot_neuron_session_histograms(behavior_glm_input)

In [ ]:
beh = behavior_glm_input
beh

In [ ]:
# Install UMAP if needed (uncomment if not installed)
# !pip install umap-learn

import umap
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ignore features
ign_features = [
    'trial_outcome',
    'cue',
    'choice_R1',
    'choice_R2',
    'global_t',
]
data_clean = beh.drop(columns=ign_features)

print(f"Data shape: {data_clean.shape}")
print(f"Number of features: {data_clean.shape[1]}")
print(f"Features: {data_clean.columns}")
print(f"Number of samples: {data_clean.shape[0]}")

# Standardize the data
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_clean)

# PCA for comparison
print("\n=== PCA Analysis ===")
pca = PCA(n_components=min(10, data_clean.shape[1]))
pca_embedding = pca.fit_transform(data_scaled)

print(f"PCA explained variance ratio (first 3 components): {pca.explained_variance_ratio_[:3]}")
print(f"PCA cumulative variance (first 3 components): {np.cumsum(pca.explained_variance_ratio_[:3])[-1]:.3f}")
print(f"PCA cumulative variance (all components): {np.cumsum(pca.explained_variance_ratio_)}")

# UMAP embedding
print("\n=== UMAP Analysis ===")
print("Fitting UMAP (this may take a few minutes)...")
reducer = umap.UMAP(
    n_components=3,
    n_neighbors=10,
    min_dist=0.1,
    metric='euclidean',
    random_state=41,
    verbose=True
)
umap_embedding = reducer.fit_transform(data_scaled)
print("UMAP fitting complete! Shape ", umap_embedding.shape)

# Create color mapping by session
session_colors = data_clean.index.get_level_values('session_id')
print(f"\nUnique sessions: {session_colors.unique()}")

In [ ]:
umap_embedding

In [ ]:

# Function to create 3D visualizations colored by specific features
def plot_umap_pca_colored_by(labels_dataframe, color_by=None, output_subdir='umap_pca_by_feature',
                             mask=None):
    """
    Create side-by-side UMAP and PCA 3D visualizations colored by specific features.
    
    Parameters:
    -----------
    labels_dataframe : pd.DataFrame
        DataFrame containing features/labels to color by. Must share index with embeddings.
    color_by : str, list, or None
        Feature name(s) to color by. If None, uses all columns. If list, iterates through all.
        If 'index', colors by the dataframe index.
    output_subdir : str
        Subdirectory name within output_dir to save HTML files.
    """
    import os
    
    # Create output directory
    viz_dir = f"{output_dir}/{output_subdir}/"
    os.makedirs(viz_dir, exist_ok=True)
    
    # Determine which features to plot
    if color_by is None:
        features_to_plot = labels_dataframe.columns.tolist()
    elif color_by == 'index':
        features_to_plot = ['index']
    elif isinstance(color_by, str):
        features_to_plot = [color_by]
    elif isinstance(color_by, list):
        features_to_plot = color_by
    else:
        raise ValueError("color_by must be None, 'index', a string, or a list of strings")
    
    print(f"Creating visualizations for {len(features_to_plot)} features:\n{features_to_plot}\n")
    print(labels_dataframe.shape)
    print(umap_embedding.shape)
    # Create a visualization for each feature
    for feature in features_to_plot:
        print(f"Creating visualization for: {feature}")
        
        # Get feature values for coloring
        if feature == 'index':
            feature_values = labels_dataframe.index.get_level_values(0).values
            feature_name = 'Index'
        else:
            feature_values = labels_dataframe[feature].values
            feature_name = feature
        
        # Create figure with side-by-side 3D plots
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=(f'UMAP 3D - colored by {feature_name}', 
                          f'PCA 3D - colored by {feature_name}'),
            specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
            horizontal_spacing=0.05
        )
        
        if mask is None:
            mask = np.ones_like(feature_values).astype(bool)
        
        # Add UMAP 3D scatter
        fig.add_trace(
            go.Scatter3d(
                x=umap_embedding[mask, 0],
                y=umap_embedding[mask, 1],
                z=umap_embedding[mask, 2],
                mode='markers',
                marker=dict(
                    size=2, 
                    color=feature_values[mask], 
                    colorscale='Viridis',
                    showscale=True, 
                    colorbar=dict(title=feature_name, x=0.46, len=0.8),
                    opacity=0.6
                ),
                text=[f'{feature_name}: {v:.2f}' if isinstance(v, (int, float)) else f'{feature_name}: {v}' 
                      for v in feature_values[mask]],
                hovertemplate='<b>%{text}</b><br>UMAP1: %{x:.2f}<br>UMAP2: %{y:.2f}<br>UMAP3: %{z:.2f}<extra></extra>'
            ),
            row=1, col=1
        )
        
        # Add PCA 3D scatter
        fig.add_trace(
            go.Scatter3d(
                x=pca_embedding[mask, 0],
                y=pca_embedding[mask, 1],
                z=pca_embedding[mask, 2],
                mode='markers',
                marker=dict(
                    size=2,
                    color=feature_values[mask],
                    colorscale='Viridis',
                    showscale=False,
                    opacity=0.6
                ),
                text=[f'{feature_name}: {v:.2f}' if isinstance(v, (int, float)) else f'{feature_name}: {v}' 
                      for v in feature_values[mask]],
                hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>PC3: %{z:.2f}<extra></extra>'
            ),
            row=1, col=2
        )
        
        # Update layout
        fig.update_layout(
            title_text=f'3D Embedding Comparison - Colored by {feature_name}',
            height=600,
            width=1400,
            showlegend=False,
            font=dict(size=10)
        )
        
        # Update scenes with labels
        fig.update_scenes(
            xaxis_title='Dimension 1',
            yaxis_title='Dimension 2',
            zaxis_title='Dimension 3',
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.3))
        )
        
        # Save to HTML
        sanitized_feature = feature_name.replace('/', '_').replace(' ', '_').replace(':', '_')
        html_file = f"{viz_dir}{sanitized_feature}_umap_pca_3d.html"
        fig.write_html(html_file)
        print(f"  ✓ Saved to: {html_file}")
    
    print(f"\n✓ All visualizations saved to: {viz_dir}")
    print(f"Total files created: {len(features_to_plot)}")
    
    return viz_dir



In [ ]:
# check the general shape of clusters before moving on
plot_umap_pca_colored_by(data_clean, None, )

In [ ]:
# # K-means clustering and elbow curve analysis
# from sklearn.cluster import KMeans
# from sklearn.metrics import silhouette_score

# print("=== K-means Clustering Analysis ===\n")

# # Sample 50,000 points if dataset is larger (for computational efficiency)
# if len(data_scaled) > 50000:
#     sample_indices = np.random.choice(len(data_scaled), size=50000, replace=False)
#     data_scaled_sample = data_scaled[sample_indices]
#     print(f"Sampled {len(data_scaled_sample)} points from {len(data_scaled)} total")
# else:
#     data_scaled_sample = data_scaled
#     print(f"Using all {len(data_scaled)} points")

# # Fit k-means for different numbers of clusters
# # Start from k=2 since silhouette_score requires at least 2 clusters
# k_values = range(2, 14)
# inertias = []
# silhouette_scores = []

# for k in k_values:
#     kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
#     cluster_labels = kmeans.fit_predict(data_scaled_sample)
#     inertias.append(kmeans.inertia_)
    
#     # Compute silhouette score (requires at least 2 clusters and fewer than n_samples)
#     if 1 < k < len(data_scaled_sample):
#         sil_score = silhouette_score(data_scaled_sample, cluster_labels)
#         silhouette_scores.append(sil_score)
#     else:
#         silhouette_scores.append(None)
    
#     # Format silhouette score string
#     sil_str = f"{silhouette_scores[-1]:.3f}" if silhouette_scores[-1] is not None else "N/A"
#     print(f"  k={k}: inertia={kmeans.inertia_:.2f}, silhouette={sil_str}")

# # Create elbow curve visualization
# fig = make_subplots(
#     rows=1, cols=2,
#     subplot_titles=('Elbow Curve (Inertia)', 'Silhouette Score'),
#     specs=[[{'secondary_y': False}, {'secondary_y': False}]]
# )

# # Plot 1: Elbow curve
# fig.add_trace(
#     go.Scatter(
#         x=list(k_values),
#         y=inertias,
#         mode='lines+markers',
#         name='Inertia',
#         line=dict(color='blue', width=2),
#         marker=dict(size=8)
#     ),
#     row=1, col=1
# )

# # Plot 2: Silhouette scores
# fig.add_trace(
#     go.Scatter(
#         x=list(k_values),
#         y=silhouette_scores,
#         mode='lines+markers',
#         name='Silhouette Score',
#         line=dict(color='green', width=2),
#         marker=dict(size=8)
#     ),
#     row=1, col=2
# )

# # Update layout
# fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=1)
# fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=2)
# fig.update_yaxes(title_text="Inertia", row=1, col=1)
# fig.update_yaxes(title_text="Silhouette Score", row=1, col=2)

# fig.update_layout(
#     title_text="K-means Elbow Curve and Silhouette Analysis",
#     height=500,
#     width=1200,
#     showlegend=True,
#     font=dict(size=11)
# )

# # Save elbow curve
# elbow_html = f"{viz_output_dir}kmeans_elbow_curve.html"
# fig.write_html(elbow_html)
# print(f"\n✓ Elbow curve saved to: {elbow_html}")

# # Print clustering summary
# print("\n=== Clustering Summary ===")
# print(f"Total samples used for clustering: {len(data_scaled_sample)}")
# print(f"Total features: {data_scaled_sample.shape[1]}")
# print(f"\nSilhouette scores by k:")
# for k, sil in zip(k_values, silhouette_scores):
#     if sil is not None:
#         print(f"  k={k}: {sil:.4f}")
#     else:
#         print(f"  k={k}: N/A (insufficient clusters or too many)")

# # Find and report optimal k
# valid_scores = [(k, sil) for k, sil in zip(k_values, silhouette_scores) if sil is not None]
# if valid_scores:
#     optimal_k, optimal_score = max(valid_scores, key=lambda x: x[1])
#     print(f"\n✓ Optimal k (highest silhouette score): k={optimal_k} with score={optimal_score:.4f}")

In [ ]:
# Create k-means clustering with k=9 and build comprehensive labels dataframe
from sklearn.cluster import KMeans

# Run k-means with k=9
k_optimal = 8
kmeans_final = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(data_scaled)

In [ ]:

print("=== Creating Labels Dataframe ===\n")

# Create labels dataframe starting with cluster assignments
labels_df = pd.DataFrame({'kmeans_cluster': cluster_labels}, index=data_clean.index)

# Add the previously dropped behavioral features
for feat in ign_features:
    if feat in beh.columns:
        labels_df[feat] = beh.loc[:, feat]
        print(f"Added feature: {feat}")

print(f"\n✓ Labels dataframe created with shape: {labels_df.shape}")
print(f"✓ Columns: {list(labels_df.columns)}")
(labels_df)


In [ ]:
plt.hist(labels_df['kmeans_cluster'], bins=np.arange(-0.5, k_optimal + 1.5, 1), rwidth=0.8)
plt.xlabel("K-means Cluster")
plt.ylabel("Count")

In [ ]:
fr_first7 = fr[nonan_mask.values].loc[:7]
fr_first7

In [ ]:
mask = fr_first7['Unit0015'] > 50
plot_umap_pca_colored_by(fr_first7, color_by='Unit0015', output_subdir='umap_neuron_labels', mask=mask)
fr_first7

In [ ]:
# plot_umap_pca_colored_by(labels_df, color_by=None, output_subdir='umap_trialwise_labels')

plot_umap_pca_colored_by(data_clean, color_by=None, output_subdir='umap_fitted_labels')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist


cluster_ids = np.sort(labels_df.kmeans_cluster.unique())
neurons = fr_first7.columns

# Preallocate: rows = neurons, cols = clusters
heatmap = np.zeros((len(neurons), len(cluster_ids)))

for n, unit_col in enumerate(neurons):
    fr_unit = fr_first7[unit_col]
    if (fr_unit == 0).all():
        continue
    # zscore
    fr_unit = (fr_unit-fr_unit.mean()) /fr_unit.std()
    if unit_col == 'Unit0015':
        print(fr_unit)

    for j, cid in enumerate(cluster_ids):
        cluster_mask = labels_df.kmeans_cluster == cid
        heatmap[n, j] = fr_unit[cluster_mask.values].mean()
        if unit_col == 'Unit0015':
            print(cid, fr_unit[cluster_mask.values].mean())

row_dist = pdist(heatmap, metric="euclidean")
row_linkage = linkage(row_dist, method="average")
row_order = leaves_list(row_linkage)

# col_dist = pdist(heatmap.T, metric="euclidean")
# col_linkage = linkage(col_dist, method="average")
# col_order = leaves_list(col_linkage)

heatmap_ord = heatmap[row_order]#[:, col_order]
neurons_ord = neurons[row_order]
cluster_ids_ord = cluster_ids# [col_order]


plt.figure(figsize=(.6 * len(cluster_ids_ord), 0.15 * len(neurons_ord)))

im = plt.imshow(
    heatmap_ord,
    aspect="auto",
    interpolation="nearest",
    cmap="viridis",
    norm=Normalize(
        vmin=np.nanmin(heatmap_ord),
        vmax=np.nanmax(heatmap_ord)
    )
)

plt.colorbar(im, label="Mean firing rate (Hz)")

plt.xticks(
    ticks=np.arange(len(cluster_ids_ord)),
    labels=cluster_ids_ord,
    rotation=45,
    ha="right"
)
plt.yticks(
    ticks=np.arange(len(neurons_ord)),
    labels=neurons_ord
)

plt.xlabel("K-means cluster (hierarchically ordered)")
plt.ylabel("Neuron (hierarchically ordered)")
plt.title("Mean firing rate per neuron per cluster (linkage clustering)")

plt.tight_layout()
plt.show()
cluster_ids_ord

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

neuron_id = 15
neuron_col = fr_first7.columns[neuron_id]

fr_neuron = fr_first7[neuron_col]

# z-score identically to heatmap
fr_neuron = (fr_neuron - fr_neuron.mean()) / fr_neuron.std()

cluster_data = []
cluster_labels = []

for cid in np.sort(labels_df.kmeans_cluster.unique()):
    cluster_mask = labels_df.kmeans_cluster == cid
    cluster_fr = fr_neuron[cluster_mask.values]
    cluster_data.append(cluster_fr)
    cluster_labels.append(f"C{cid}")

plt.figure(figsize=(12, 6))

parts = plt.violinplot(
    cluster_data,
    positions=np.arange(len(cluster_labels)),
    showmeans=True,
    showmedians=True,
    widths=0.7
)

for pc in parts['bodies']:
    pc.set_facecolor('steelblue')
    pc.set_alpha(0.7)

plt.xticks(np.arange(len(cluster_labels)), cluster_labels)
plt.xlabel("K-means cluster (raw order)")
plt.ylabel("Z-scored firing rate")
plt.title(f"Firing rate distribution for {neuron_col}")
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\n=== {neuron_col} Statistics by Cluster ===")
for cid in np.sort(labels_df.kmeans_cluster.unique()):
    cluster_mask = labels_df.kmeans_cluster == cid
    cluster_fr = fr_neuron[cluster_mask.values]
    print(
        f"Cluster {cid}: "
        f"mean={cluster_fr.mean():.2f}, "
        f"median={cluster_fr.median():.2f}, "
        f"std={cluster_fr.std():.2f}, "
        f"n={len(cluster_fr)}"
    )


In [ ]:
# ORDER OF THIS IS WRONG? CLUSTER IDS SOMEHOW MIXED UP BETWEEN HEATMAP AND VIOLIN PLOTS

In [ ]:
# # confirm that t- events are in the right region using bodycam

# camera = 'bodycam'
# plt.close("all")
# for s_id in t0_events.index.unique('session_id'):
#     if s_id <8:
#         continue
    
#     data = session_modality_from_nas(session_dirs[s_id], f"{camera}_packages")
#     for t0 in t0_events.loc[s_id].index[::19]:
#         print(t0_events.loc[s_id, t0])
        
#         fig, axes = plt.subplots(ncols=6, figsize=(20,3))
#         fig.suptitle(f'Session {s_id}, t0: {t0}', fontsize=16)
#         for i, delta in enumerate(range(-12, 12, 4)):
#             closest_frame = data.iloc[(data[f'{camera}_image_ephys_timestamp'] - t0).abs().argsort()[0] +delta]
#             frame_id = int(closest_frame.loc[f'{camera}_image_id']) 
#             ephys_t = closest_frame.loc[f'{camera}_image_ephys_timestamp'] + (delta * 40000)
            
#             closest_beh_t = (behavior.xs(s_id, level='session_id').loc[:, 'from_ephys_timestamp'] - ephys_t).abs().argsort().values[0]
#             closest_beh_frame = behavior.xs(s_id, level='session_id').iloc[closest_beh_t]
            
#             # print(f"Frame id: {frame_id}, ephys time: {ephys_t}, closest behavior time: {closest_beh_t}, frame position: {closest_beh_frame['frame_position']}")
#             # print(f"Looking for t0: {t0}, closest frame id: {frame_id}, ephys time: {ephys_t}, diff: {ephys_t - t0}")
#             # print(f"Position at t0: {t0_events.loc[(s_id, t0), 'x_position']}")
#             title = f'x: {closest_beh_frame['frame_position']:.2f}cm, \nt_diff: {(ephys_t - t0)/1e6:.3f} us, \ndelta={delta}'
            
#             frame_key = f'frame_{frame_id:06d}'

#             img = session_modality_from_nas(session_dirs[s_id], key=f"{camera}_frames", columns=[frame_key])
#             axes[i].imshow(img[0])
#             axes[i].axis('off')
#             axes[i].set_title(title, fontsize=8)
        
#         plt.tight_layout()
#         plt.show()
#         # break
#     break
        

In [ ]:
neurons_ord